In [ ]:
!pip install diffusers transformers accelerate torch safetensors

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())

True
1


In [ ]:
import torch
from diffusers import StableDiffusionPipeline

model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)

pipe = pipe.to("cuda")

print("Model Loaded Successfully!")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Model Loaded Successfully!


In [ ]:
prompt = """
Traditional Indian bridal lehenga,
royal red and gold embroidery,
luxury fashion,
highly detailed,
fashion photography
"""

image = pipe(prompt).images[0]

image.save("bridal_lehenga.png")

image

In [3]:
import torch
import cv2
import numpy as np
from PIL import Image
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

def process_sketch_to_garment(sketch_path, prompt, output_path):
    """
    Takes a fashion sketch, applies Canny edge detection,
    and generates a photorealistic garment using ControlNet.
    """
    print("Loading sketch image...")
    # Load and process the source sketch
    original_image = Image.open(sketch_path).convert("RGB")
    image_np = np.array(original_image)

    # Extract edges to create the control map
    low_threshold = 100
    high_threshold = 200
    canny_image = cv2.Canny(image_np, low_threshold, high_threshold)
    canny_image = canny_image[:, :, None]
    canny_image = np.concatenate([canny_image, canny_image, canny_image], axis=2)
    control_image = Image.fromarray(canny_image)

    print("Loading ControlNet and Stable Diffusion models...")
    # Load ControlNet model for Canny edges
    controlnet = ControlNetModel.from_pretrained(
        "lllyasviel/sd-controlnet-canny",
        torch_dtype=torch.float16
    )

    # Load base Stable Diffusion pipeline with ControlNet
    pipe = StableDiffusionControlNetPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        controlnet=controlnet,
        torch_dtype=torch.float16
    )

    # Optimize scheduler for faster inference
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

    # Enable memory savings if running on limited GPU
    pipe.enable_model_cpu_offload()

    print(f"Generating fashion design for prompt: '{prompt}'...")
    # Generate the image
    # We use a conditioning scale of 0.8 to balance structure and fabric realism
    output = pipe(
        prompt,
        image=control_image,
        negative_prompt="poorly drawn, deformed, ugly, bad anatomy, low resolution",
        num_inference_steps=30,
        controlnet_conditioning_scale=0.8
    ).images[0]

    # Save the output
    output.save(output_path)
    print(f"Success! Output saved to {output_path}")

if __name__ == "__main__":
    # Example usage for submission testing
    input_sketch = "fashion_sketch_input.jpg" # NOTE: Ensure this file exists in your folder
    text_prompt = "high fashion runway photography, elegant emerald green velvet dress, intricate gold embroidery, photorealistic, 8k"
    output_render = "final_garment_output.png"

    # process_sketch_to_garment(input_sketch, text_prompt, output_render)

In [2]:
import warnings
import logging
# Suppress user warnings and logging clutter
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("diffusers").setLevel(logging.ERROR)

In [5]:
if __name__ == "__main__":
    import os
    from PIL import Image, ImageDraw

    input_sketch = "fashion_sketch_input.jpg"
    text_prompt = "high fashion runway photography, elegant emerald green velvet dress, intricate gold embroidery, photorealistic, 8k"
    output_render = "final_garment_output.png"

    # AUTOMATIC FIX: If you don't have an image, the script creates a basic dress outline for you
    if not os.path.exists(input_sketch):
        print(f"'{input_sketch}' not found. Automatically creating a mock fashion sketch for you...")
        # Create a 512x512 white image
        mock_sketch = Image.new("RGB", (512, 512), "white")
        draw = ImageDraw.Draw(mock_sketch)
        # Draw a simple geometric dress silhouette line
        draw.polygon([(256, 80), (180, 200), (130, 450), (382, 450), (332, 200)], outline="black", width=5)
        mock_sketch.save(input_sketch)
        print(f"Created {input_sketch} successfully!")

    # Run the generation pipeline
    process_sketch_to_garment(input_sketch, text_prompt, output_render)

'fashion_sketch_input.jpg' not found. Automatically creating a mock fashion sketch for you...
Created fashion_sketch_input.jpg successfully!
Loading sketch image...
Loading ControlNet and Stable Diffusion models...


config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

Generating fashion design for prompt: 'high fashion runway photography, elegant emerald green velvet dress, intricate gold embroidery, photorealistic, 8k'...


  0%|          | 0/30 [00:00<?, ?it/s]

Success! Output saved to final_garment_output.png
